In [0]:
from datetime import datetime, timedelta
from pyspark.sql.types import StringType
import json

# Date Utility to generate Date data

def date_data(start_run_dt: str = '20230101',num_years: int = 1) -> list:
    _data = []
    start_date = datetime.strptime(start_run_dt, '%Y%m%d')
    _data.append([start_date.strftime('%Y-%m-%d'),start_date.strftime('%d'),start_date.strftime('%m'),start_date.strftime('%Y'),start_date.strftime('%A')])
    for i in range(0,num_years*365):
        _next_date = start_date + timedelta(days=i+1)
        _data.append([_next_date.strftime('%Y-%m-%d'),_next_date.strftime('%d'),_next_date.strftime('%m'),_next_date.strftime('%Y'),_next_date.strftime('%A')])

    return _data

def get_rundate():
    try:
        with open('/Workspace/Users/mohanakapa@outlook.com/databricks/DeltaLakeWarehouse/config/run_config.txt','r') as f:
            data = json.load(f)
            return data['rundate']
    except Exception as e:
        print(e)
        return '19000101'

In [0]:
def insert_log(schema_name: str, table_name: str, max_timestamp,rundate: str) -> bool:
  """
  This function inserts a log record into the table 'log' in the schema 'schema_name'
  """
  try:
      _data = [[schema_name, table_name, max_timestamp,rundate]]
      _columns = ['schema_name', 'table_name', 'max_timestamp','rundate']
      df = spark.createDataFrame(_data, _columns)

      #Create necessary Columns
      df_processed = df.selectExpr('schema_name','table_name','to_timestamp(max_timestamp) as max_timestamp','rundate','current_timestamp() as insert_dt')

      #write into Job Control
      df_processed.write.format('delta').mode('append').saveAsTable('warehouse.edw.job_control')
      return True
  except Exception as e:
      print(e)

In [0]:
insert_log('dummy','dummy','2023-01-01','202020')

In [0]:
_data = [['schema_name', 'table_name', 'max_timestamp','rundate']]
_columns = ['schema_name', 'table_name', 'max_timestamp','rundate']
df = spark.createDataFrame(_data, _columns)

#Create necessary Columns
df_processed = df.selectExpr('schema_name','table_name','to_timestamp(max_timestamp) as max_timestamp','rundate','current_timestamp() as insert_dt')

In [0]:
rundate = '20220101'
schema_name = 'warehouse.edw_ld'
table_name = 'dim_date_ld'

In [0]:
import datetime
insert_log(schema_name,table_name,datetime.datetime.now(),rundate)